# Mobile Money Agent Network: Data Audit, Cleaning and Summary
**Analyst:** Stanley Wanyoike Ngugi

Audit, clean and summarise an unreliable transaction extract, merge it with the agent register, and document every decision. All cleaning rules live in `cleaning_pipeline.py`.

In [1]:
import os
if not os.path.exists("cleaning_pipeline.py"):
    !git clone https://github.com/SWNgugi/plp-da-a3-duka-sales-and-agent-cleanup.git
    %cd plp-da-a3-duka-sales-and-agent-cleanup

import numpy as np
import pandas as pd
from cleaning_pipeline import clean_agent_data

if not os.path.exists("agent_transactions.csv"):
    !python generate_data.py
os.makedirs("outputs", exist_ok=True)

fatal: destination path 'plp-da-a3-duka-sales-and-agent-cleanup' already exists and is not an empty directory.
/content/plp-da-a3-duka-sales-and-agent-cleanup


## Task 1: Load and audit
The raw files are loaded unchanged. pandas reads "N/A" as missing by default, so I check `isna()` as well as invalid strings. The audit table is calculated from the raw data with code.

In [2]:
txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")
print("transactions:", txns.shape, "| agents:", agents.shape)
print(txns.dtypes)
txns.head()

transactions: (612, 5) | agents: (20, 3)
txn_id       object
agent_id     object
txn_type     object
amount      float64
txn_date     object
dtype: object


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [3]:
amt_num = pd.to_numeric(txns["amount"], errors="coerce")
q1, q3 = amt_num.quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)

dates = pd.to_datetime(txns["txn_date"].astype(str).str.replace("/", "-"),
                       format="%Y-%m-%d", errors="coerce")
onboarded = txns["agent_id"].map(agents.set_index("agent_id")["onboarded"].pipe(pd.to_datetime))
bad_timeliness = int(((dates < onboarded) | (dates > pd.Timestamp.today())).sum())

audit = pd.DataFrame([
    ["Duplicate transactions (repeated txn_id)", "Uniqueness", "txn_id", int(txns.duplicated("txn_id").sum())],
    ["Inconsistent spelling/case/whitespace in txn_type", "Consistency", "txn_type", int((~txns["txn_type"].isin(["Deposit", "Withdrawal"])).sum())],
    ["Missing amount (blank or 'N/A')", "Completeness", "amount", int(amt_num.isna().sum())],
    ["Dates written as YYYY/MM/DD instead of YYYY-MM-DD", "Validity", "txn_date", int(txns["txn_date"].astype(str).str.contains("/").sum())],
    ["Identical round amount of 250,000 (suspected placeholder/cap)", "Accuracy", "amount", int((amt_num == 250000).sum())],
    ["Amounts above the IQR upper fence", "Accuracy", "amount", int((amt_num > upper_fence).sum())],
    ["agent_id not in agent register", "Consistency", "agent_id", int((~txns["agent_id"].isin(agents["agent_id"])).sum())],
    ["Dates before agent onboarding or in the future", "Timeliness", "txn_date", bad_timeliness],
], columns=["issue", "dimension", "column", "rows_affected"])
audit

,issue,dimension,column,rows_affected
0,Duplicate transactions (repeated txn_id),Uniqueness,txn_id,12
1,Inconsistent spelling/case/whitespace in txn_type,Consistency,txn_type,384
2,Missing amount (blank or 'N/A'),Completeness,amount,22
3,Dates written as YYYY/MM/DD instead of YYYY-MM-DD,Validity,txn_date,5
4,"Identical round amount of 250,000 (suspected p...",Accuracy,amount,5
5,Amounts above the IQR upper fence,Accuracy,amount,40
6,agent_id not in agent register,Consistency,agent_id,8
7,Dates before agent onboarding or in the future,Timeliness,txn_date,0


## Tasks 2 to 5: Cleaning (uniqueness, consistency, validity, completeness)
All rules run through `clean_agent_data()`, and every rule is written to the cleaning log below.

In [4]:
clean, log = clean_agent_data(txns, agents)
log

,step,rule,rows_affected,action
0,1,Input,612,612 raw transaction rows loaded
1,2,Duplicate txn_id,12,"Removed, first occurrence kept"
2,3,Inconsistent txn_type,376,"Trimmed, lower-cased and mapped to Deposit/Wit..."
3,4,Missing/invalid amount,20,Converted to numeric; invalid text became NaN
4,5,Unparseable txn_date,0,Slash dates converted to ISO; unparseable set ...
5,6,Imputed amount,20,Filled with median amount of same txn_type; am...
6,7,Outlier amount (IQR),40,"Flagged, not removed; fences -1,928 to 6,084"
7,8,Unknown agent_id,7,Kept; county set to 'Unknown' so totals still ...
8,9,Output,600,600 clean rows


### Uniqueness
12 duplicate transactions were removed (612 → 600 rows). I de-duplicate on `txn_id`, not on the whole row: 11 duplicates are identical rows, but one copy had its `agent_id` overwritten with an unknown agent, so a whole-row check would miss it.

### Consistency
`txn_type` had five spellings (e.g. "DEPOSIT ", "deposit", "withdrawal"). After trimming whitespace and lower-casing, they map to exactly two values: Deposit and Withdrawal.

### Validity and types
`amount` is converted to numeric and `txn_date` to datetime. 5 dates written as YYYY/MM/DD were converted to ISO format, and all 600 dates now parse.

### Completeness
20 transactions (after de-duplication) had a blank or "N/A" amount. I impute the median amount of the same transaction type and set `amount_imputed = True`. Reasons: the transactions did happen, so dropping them would understate volumes. The median is robust to the skewed amounts, and the flag lets anyone exclude imputed values. Value totals for these rows are estimates; transaction counts are unaffected.

## Task 6: Outliers (IQR method)
Amounts outside Q1 − 1.5·IQR and Q3 + 1.5·IQR are flagged in `is_outlier`. Fences are calculated on real (non-imputed) amounts only. Nothing is deleted.

In [5]:
print("Flagged outliers:", clean["is_outlier"].sum())
suspect = clean.loc[clean["amount"] == 250000, ["txn_id", "agent_id", "county", "txn_type", "txn_date", "is_outlier"]]
print("Rows with exactly KES 250,000:", len(suspect))
print("Their share of total value: {:.1%}".format(250000 * len(suspect) / clean["amount"].sum()))
suspect

Flagged outliers: 40
Rows with exactly KES 250,000: 5
Their share of total value: 46.5%


,txn_id,agent_id,county,txn_type,txn_date,is_outlier
306,TX00307,AG012,Kisumu,Deposit,2026-03-02,True
315,TX00316,AG014,Nairobi,Deposit,2026-04-01,True
424,TX00425,AG005,Mombasa,Withdrawal,2026-03-02,True
463,TX00464,AG009,Nakuru,Withdrawal,2026-04-11,True
594,TX00595,AG016,Kiambu,Deposit,2026-03-11,True


### Recommendation on the KES 250,000 values
Five transactions carry exactly KES 250,000, about 35 times the upper IQR fence and 46.5% of total transaction value (KES 1,250,000 of KES 2,688,596). An identical round value repeated five times looks like a system cap, test value or placeholder rather than genuine customer activity. I recommend keeping them flagged, excluding them from averages and headline totals until the agents or the source system confirm them, and then either correcting or removing them. The other 35 flagged amounts (KES 6,158 and above) are plausible large transactions in a right-skewed distribution and are kept.

## Task 7: Merge with the agent register
A left join keeps every transaction. Unknown agents get county "Unknown", so no valid transaction is lost and totals reconcile.

In [6]:
print("Raw rows:", len(txns))
print("After removing duplicates:", len(txns.drop_duplicates("txn_id")))
print("After merge (clean):", len(clean))
print("Transactions with unknown agents:", int(clean["unknown_agent"].sum()))
print("Unknown agent ids:", clean.loc[clean["unknown_agent"], "agent_id"].unique())

Raw rows: 612
After removing duplicates: 600
After merge (clean): 600
Transactions with unknown agents: 7
Unknown agent ids: ['AG099']


## Task 8: Summaries

In [7]:
county_summary = (clean.groupby(["county", "txn_type"])["amount"]
                  .agg(total_value="sum", n_transactions="count", avg_value="mean")
                  .round(0))
county_summary

total_value  n_transactions  avg_value
county  txn_type                                          
Kiambu  Deposit        334737.0              43     7785.0
        Withdrawal      77024.0              27     2853.0
Kisumu  Deposit        436502.0              74     5899.0
        Withdrawal      76348.0              37     2063.0
Mombasa Deposit        149094.0              72     2071.0
        Withdrawal     368592.0              61     6042.0
Nairobi Deposit        543698.0             118     4608.0
        Withdrawal     161932.0              62     2612.0
Nakuru  Deposit        184944.0              60     3082.0
        Withdrawal     328568.0              39     8425.0
Unknown Deposit         25279.0               6     4213.0
        Withdrawal       1877.0               1     1877.0

In [8]:
weekly = clean.assign(week_start=clean["txn_date"].dt.to_period("W-SUN")
                      .dt.start_time.dt.strftime("%Y-%m-%d"))
weekly_pivot = weekly.pivot_table(index="week_start", columns="county", values="amount",
                                  aggfunc="sum", fill_value=0).round(0)
weekly_pivot

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week_start,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2770.0,0.0
2026-03-02,18317.0,287933.0,285696.0,68967.0,56574.0,0.0
2026-03-09,267746.0,23186.0,30504.0,52888.0,28730.0,0.0
2026-03-16,22054.0,28126.0,25172.0,43134.0,39795.0,1639.0
2026-03-23,11504.0,27077.0,17050.0,41400.0,18972.0,0.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06,15030.0,15341.0,47120.0,56814.0,276609.0,9824.0
2026-04-13,12265.0,29470.0,26730.0,56406.0,26180.0,0.0
2026-04-20,24507.0,37439.0,16687.0,65135.0,30862.0,1877.0


In [9]:
total_all = clean["amount"].sum()
total_ex = clean.loc[clean["amount"] != 250000, "amount"].sum()
print(f"Total value, all rows: KES {total_all:,.0f}")
print(f"Total value, excluding the 250,000 rows: KES {total_ex:,.0f}")

clean.to_csv("outputs/agent_transactions_clean.csv", index=False)
print(pd.read_csv("outputs/agent_transactions_clean.csv").head())

Total value, all rows: KES 2,688,596
Total value, excluding the 250,000 rows: KES 1,438,596
    txn_id agent_id    txn_type  amount    txn_date  amount_imputed  \
0  TX00001    AG009     Deposit   353.0  2026-03-03           False   
1  TX00002    AG020     Deposit  1238.0  2026-04-27           False   
2  TX00003    AG008     Deposit  9129.0  2026-04-29           False   
3  TX00004    AG009     Deposit   652.0  2026-04-24           False   
4  TX00005    AG009  Withdrawal   478.0  2026-04-17           False   

   is_outlier   county  unknown_agent  
0       False   Nakuru          False  
1       False  Mombasa          False  
2        True   Kisumu          False  
3       False   Nakuru          False  
4       False   Nakuru          False  


## Data Quality Report
**Scope:** 612 raw transaction rows and 20 agents became 600 clean rows (12 removed). Transactions cover 1 March to 29 April 2026.

**Completeness.** 22 raw rows (20 after de-duplication) had a blank or "N/A" amount. They were imputed with the median amount for their transaction type and flagged in `amount_imputed`, so value totals for those rows are estimates.

**Uniqueness.** 12 rows repeated an existing `txn_id` and were removed, keeping the first occurrence. Matching on `txn_id` was needed because one duplicate differed in `agent_id`.

**Validity.** `amount` was stored with text such as "N/A" and is now numeric. 5 dates used YYYY/MM/DD and were converted to ISO format; all 600 dates now parse.

**Consistency.** `txn_type` had five spellings across 384 of 612 raw rows and is now exactly Deposit or Withdrawal. 8 raw rows (7 after de-duplication) reference agent AG099, which is not in the register; they are kept with county "Unknown" and flagged in `unknown_agent`.

**Accuracy.** Five amounts equal exactly KES 250,000 (46.5% of total value) and are flagged as suspected placeholders or caps, together with 35 other large amounts (40 flagged, IQR fences −1,928 to 6,084). Nothing was deleted. The 250,000 rows should be verified with the source system before they are used in averages or totals.

**Timeliness.** No transaction falls in the future or before its agent was onboarded. The extract covers only 60 days, so it cannot support longer-term trend conclusions.

**Privacy.** If the extract also contained customer phone numbers, the phone number would be personal data under the Kenya Data Protection Act 2019, because it identifies a person. Name, national ID, and a transaction history that can be linked to a phone number would be personal data too. I would protect them by collecting only what the analysis needs (data minimisation), replacing phone numbers with salted hashed IDs before analysis, restricting access to named staff, encrypting the data in storage and transfer, setting a retention period, and never committing raw data with phone numbers to GitHub. Any breach would need to be reported to the Office of the Data Protection Commissioner. (Check the exact section numbers against the Act before you cite them.)